**Author:** Salvador Navas  
**Date:** 2025-06-27

### OGIMET — WMO SYNOP Observations Worldwide

**OGIMET** (https://www.ogimet.com) provides historical and near real-time meteorological
observations from **WMO SYNOP stations** worldwide, aggregated from official WMO bulletins.

#### 🌐 What is a SYNOP station?

SYNOP (Surface Synoptic Observations) is the **WMO standard** for land surface weather observations.
Stations report temperature, pressure, humidity, precipitation, wind, and cloud cover at fixed
intervals (typically every 3 or 6 hours). Each station has a 5-digit **WMO number** (e.g., 08221 = Barajas).

#### 🆚 OGIMET vs other rainfall data sources

| Source | Coverage | Density | Quality | Best for |
|--------|----------|---------|---------|----------|
| **OGIMET** | Global (WMO network) | Sparse (~10k stations worldwide) | Variable | International stations, rapid access without registration |
| **AEMET** | Spain only | Dense (~800 stations) | High (national QC) | Spanish catchments, long records |
| **ERA5** | Global | 0.25° grid (~25 km) | High (reanalysis) | Ungauged regions, spatial fields |
| **GPM IMERG** | 60°S–60°N | ~10 km grid | Moderate | Remote areas, sub-daily |

> **When to use OGIMET:** When you need global coverage at international stations outside Spain
> (e.g., Morocco, France, Portugal), rapid data access without registration, or SYNOP variables
> (cloud cover, visibility) not available elsewhere.
> For Spain specifically, **AEMET** provides denser coverage and better quality control.

#### 📋 Variables available

| Variable | Description | Unit |
|----------|-------------|------|
| `precipitation_mm` | Daily accumulated precipitation | mm |
| `tmax_celsius` / `tmin_celsius` / `tmed_celsius` | Daily temperature | °C |
| `wind_speed_kmh` | Mean wind speed | km/h |
| `humidity_mean_percent` | Mean relative humidity | % |
| `pressure_msl_hpa` | Sea-level pressure | hPa |
| `cloud_total_oktas` | Total cloud cover (0=clear, 8=overcast) | oktas |
| `sun_hours_day_before` | Sunshine duration | hours |

#### ⚠️ Azure / cloud execution note

OGIMET blocks IPs from cloud data centres (Azure, AWS, GCP). The download widget **only works from local machines**.
If running in the cloud, download data locally first and upload the CSVs to the shared workspace.

#### 🔗 Access: https://www.ogimet.com


In [1]:
import os
from pathlib import Path

from pyhydra.data_sources.rainfall import OGIMETDownloader
from pyhydra.data_sources.rainfall import OgimetCSVLoader, get_default_ogimet_stations_csv

# Portable repo-root / data-dir resolution (local clone, Docker, Azure ML)
_cwd = Path.cwd()
_candidates = [Path('/workspace'), _cwd, *_cwd.parents]
REPO_ROOT = next(
    (p for p in _candidates if (p / 'notebooks').exists() or (p / 'pyhydra').exists()),
    _cwd,
)
DATA_DIR = Path(os.environ.get('HYDRA_DATA_DIR', str(REPO_ROOT / 'data')))

get_default_ogimet_stations_csv()


## OGIMETDownloader — Interactive widget

The widget allows you to:
- Navigate and **select stations** on the world map.
- Choose the **date range** and destination folder.
- Download series directly to disk.

> **⚠️ Only works locally.**  
> ogimet.com blocks Azure data centre IPs (error `ConnectTimeoutError`). If you are running this notebook from the Azure Jupyter environment, use the cell below to upload already-downloaded CSVs to the shared workspace at `/workspace/data/ogimet/`.

Run the download cell **only from your local machine**.

In [2]:
import os

# Azure Container Apps exposes CONTAINER_APP_NAME; not present locally.
if os.environ.get('CONTAINER_APP_NAME'):
    print(
        '⚠️  Download not available from Azure.\n'
        'ogimet.com blocks Azure datacenter IPs (ConnectTimeoutError).\n\n'
        'Recommended workflow:\n'
        '  1. Run OGIMETDownloader() on your local machine.\n'
        '  2. CSVs are saved to ~/hydra_data/ogimet/.\n'
        '  3. Upload them to the share with the manual upload cell (see below).\n'
        '  4. They will appear in /workspace/data/ogimet/ inside this Jupyter.'
    )
else:
    OGIMETDownloader()

Loading stations: 100%|##########| 14132/14132 [00:02<00:00, 7021.66it/s]


### Carga manual de CSVs al share de Azure

Si has descargado series en local, ejecuta la celda siguiente para copiarlas al share montado en `/workspace/data/ogimet/`  
(equivalente al Azure Files share `dockerdata`).  

Ajusta `LOCAL_OGIMET_DIR` a la ruta donde están tus CSVs.

In [3]:
import shutil

LOCAL_OGIMET_DIR  = Path.home() / 'hydra_data' / 'ogimet'
REMOTE_OGIMET_DIR = DATA_DIR / 'ogimet'

if not os.environ.get('CONTAINER_APP_NAME'):
    print('This cell only takes effect from the Azure Jupyter environment.')
elif not LOCAL_OGIMET_DIR.exists():
    print(f'Local folder not found: {LOCAL_OGIMET_DIR}')
else:
    REMOTE_OGIMET_DIR.mkdir(parents=True, exist_ok=True)
    copied = 0
    for f in LOCAL_OGIMET_DIR.glob('*.csv'):
        shutil.copy2(f, REMOTE_OGIMET_DIR / f.name)
        copied += 1
    print(f'Copied {copied} files  {LOCAL_OGIMET_DIR}  →  {REMOTE_OGIMET_DIR}')


This cell only takes effect from the Azure Jupyter environment.


In [4]:
loader = OgimetCSVLoader()
loader.folder_path

In [5]:
stations_df = loader.load_station_data()
stations_df

In [6]:
series_df = loader.load_series_data('tmax_celsius')
series_df.head()

In [7]:
try:
    quality_df = loader.analyze_series_quality()
    display(quality_df)
except ValueError as e:
    print(f'⚠  {e}\n   Download data first with OGIMETDownloader.')
    quality_df = None

      station_id  start_year  end_year  ...  full_months  min_value  max_value
0  MADRID_RETIRO        2024      2024  ...            0       10.3       18.0

[1 rows x 8 columns]


---
## 📋 Understanding the quality metrics

`analyze_series_quality()` returns key statistics for each station:

| Column | Meaning | Threshold for concern |
|--------|---------|----------------------|
| `missing_percent` | % of days with NaN | >5%: infill or exclude; >10%: exclude from extremes |
| `full_years` | Years with <1% missing data | Need ≥10 for reliable frequency analysis |
| `full_months` | Months with 0 missing days | Important for seasonal analysis |
| `min_value` / `max_value` | Range check | Negatives or >500 mm/day → likely data errors |

> **Tip:** Before fitting extreme value distributions, filter to years where
> `missing_percent_annual < 1%` to avoid bias in return level estimates.

After quality control, these series are ready for:
- **Spatial interpolation** → `interpolation` notebook
- **Extreme value analysis** → `extreme_value_analysis` notebook
- **Stochastic generation** → `stochastic_generation` notebook
